In [ ]:
!apt-get install openjdk-11-jdk -y

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

#Lo siguiente hay que descargar spark, pero Google collab ya lo tiene
#Lo que si hacen falta son las librerías.
#recordar que cada vez que quiero ejecutar el código tengo que copiar la ruta del archivo de nuevo, porque va a dar error


In [4]:
import os
import glob
import findspark

# Explicitly set SPARK_HOME to the standard Colab Spark installation path.
# The error suggests /usr/local/spark is the base, but spark-submit isn't found correctly there.
# This ensures SPARK_HOME is correctly set before findspark.init()
os.environ["SPARK_HOME"] = "/usr/local/spark"

# Set PYSPARK_PYTHON to the Python interpreter being used by Colab
os.environ["PYSPARK_PYTHON"] = "python3"

# Find the PySpark and Py4J paths within SPARK_HOME
spark_python_path = os.path.join(os.environ["SPARK_HOME"], "python")
py4j_zip_pattern = os.path.join(spark_python_path, "lib", "py4j-*.zip")
py4j_zip_files = glob.glob(py4j_zip_pattern)

if py4j_zip_files:
    # Set PYTHONPATH to include PySpark and Py4J
    py4j_zip_path = py4j_zip_files[0]
    os.environ["PYTHONPATH"] = f"{spark_python_path}:{py4j_zip_path}:{os.environ.get('PYTHONPATH', '')}"
else:
    print(f"Warning: Could not find py4j zip file at {py4j_zip_pattern}. PySpark might not work correctly.")

# Initialize findspark. It will use the SPARK_HOME and PYTHONPATH we just set,
# ensuring other internal configurations are made.
findspark.init()

from pyspark.sql import SparkSession
hosp = SparkSession.builder.appName("hosp").getOrCreate()
#Así creamos sesion de spark llamada hosp
hosp

FileNotFoundError: [Errno 2] No such file or directory: '/usr/local/spark/./bin/spark-submit'

In [ ]:
#Ejecutar esta celda para montar la cuenta de google drive
#Util si tngo los datos subidos en drive
#from google.colab import drive
#drive.mount('/content/drive')

In [ ]:
#ya hemos creado la sesion
#Ahora cargamos nuestro data set. No puedo decirle que agrupe cosas si no he metido los datos
#Subir dataset que esta en canvas HealthCare
#Vamos a ponerle nombre (data frame), además vamos a meter una serie de parametros.
df = hosp.read.csv('/healthcare_dataset (1).csv' ,
                   inferSchema=True,
                   header=True)
#inferSchema va a hacer una inferencia en los datos (interpretarlos)
#header inidca que tienen encabezados

#Ahora voy a mostrar el data set a ver q tiene
df.show()

In [ ]:
#Tras inferir el esquema, vamos a verlo
df.printSchema()
#nos da que tipo de datos recoje cada variable (String, integer, date...)

In [ ]:
#Mas cosas que podemos hacer. Una serie de ejemplos (se podrían hacer muchas más cosas)
#Si buscamos en internet hay un mogogllon de posibilidades
row_count=df.count()
print(row_count)

In [ ]:
#Le vamos a pedir que agrupe. vamos a hacer grupos por clave valor (este es el comando map.reduce pero con spark)
df.groupBy('gender').agg({'age':'avg'}).show()


In [ ]:
#Ahora vamos a agrupar doctores con el dinero que gana.
df.groupBy('doctor').agg({'billing amount':'avg'}).show()


In [ ]:
#Ahora cuanto ha pagado cada aseguradora
df.groupBy('Insurance Provider').agg({'billing amount':'sum'}).show()

In [ ]:
#Hay un monton más de operadores
#groupBy es le básico y sirve para agrupar
#Hay otro para agregar {agg()}, otro para filtrar {filter()}, y seleccionar {select()}

In [ ]:
#Si quiero hacer un programa para consultar el data set puedo mediante este comando que el usuario introduzca las variables
#df.groupBy().agg{()}.show()
#Aqui hago dos operaciones: estoy escogiendo el grupo y luego estoy agregando.
#Si pusiese filter, podria filtar pej los que sean mayores que x edad, los que sean grupo A de sangre...

In [ ]:
#Aqui he filtrado y me he qdado con los mayores de 18 años.
#Una vez me ha hecho ese filtro, me selecciona la columna edad y el tipo de sangre
filter=df.filter(df['age']>18).select('age','blood type')
filter.show()
#se puede usar el comando sort() para ordenar, el profe nos va a subir la hoja como guía

In [ ]:
#después de ver los datos en el dataset lo siguiente a hacer, análisis de datos, limpieza de datos y gráficos
#el 1er paso es conocer la estructura (cabecera, tipo de datos (numéricos)
#una vez se hacen reducciones de dataset pandas es útil porque te permite hacer muchas cosas con los datos, y es compatible con muchos programas. Pero no es práctico para BigData

#ahora que está evaluado el dataset podemos rellenar los datos faltantes con is.null (esto en pandas), aquí es un comando diferente
#este comando permite eliminar filas que no contengan información:
#en pandas se rellena y no se elimina, esto es porque el dataframe tiene poca información, en pyspark se trabaja con tanta información que nos podemos permitir eliminar filas que "no sean importantes" para el dataset
df_drop=df.na.drop()

#el comando show en big data la otra funcionalidad que tiene es ejecutar el rdd, aparte de mostrar las 1as 20 líneas (pero esto es en pandas)

#otra cosa es filtrar aquellos resultados que sean nulos, me crea un nuevo dataframe (df_filter)
df_filter=df.filter(df["Age"].isNull()).show()
#aquí solo hemos evaluado la columna Age, y muestra que no hay ningún valor faltante
df_filter=df.filter(df["Gender"].isNull()).show()

In [ ]:
#si no quiero eliminar toda la fila, sino rellenarla, puedo usar:
df_drop=df.na.drop()
df_fill=df.na.fill({"Age":0})
#pero vamos a hacerlo con promedio, lo dejo en formato texto para que no se ejecute. Drop elimina la celda

#se puede hacer con la media
#agg es un comando de agregación --> quiero generar contenido. Lo primero que tengo que decir es en qué columna está ({"Age"}), y lo 2o lo que quiero hacer con eso avg
#me coge el promedio y lo guarda en una nueva la tabla sólo con ese valor, será una matriz de 1 dimensión, en informática siempre se parte del 0, por lo tanto son coordenadas 0,0
#de manera que en collect tenemos que poner la posición0,0 --> collect(0,0)
#si calculo el promedio en función de rangos de edad, tendré una matriz 1-x (x=n º de rangos) y será collect(0,1({}))
promedio=df.agg({"Age":"avg"}).collect()[0][0]
df_drop=df.na.drop()
df_fill=df.na.fill({"Age":promedio})
#no sale nada porque no hay ningún comando de mostrar en pantalla .print es el comando general de python, o show que es el de pyspark
#show es ejecutar código a través de pyspark, si queires visualizar grandes cantidades de datos usar show, si es ver un valor o pocos usar print es un "comando más barato" por así decirlo

In [ ]:
#podemos ir creando columnas que me generen información a partir de otras columnas
df_coll=df.withColumn("age_plus_5", df["Age"]+5).show()
#a todas las filas va a ir sumando 5
#si le pones el mismo nombre va a sobreescribir la columna, por eso es improtante cambiar el nombre. Importante saber que es sensible a mayúsculas y a minúsculas
#la columan se añade al final, pero como no son tablas que se enseñan a la gente es simplemente para datos
#aquí se pueden poner funciones complejas de relaciones de datos: hacer agrupaciones, poner rangos, establecer promedios. Y con el comando map-reduce puedo generar datasets en función de los datos que haya generado


#si nos hemos equivocado en el nombre de la columna podemos renombrar
df_col_ren=df.withColumnRenamed("Age", "age").show()

#drop una columna
df_col_drop=df.drop("age_plus_5").show()

In [ ]:
#en lugar de trabajar con columnas, vamos a trabajar con filas
#para aádir una nueva línea y evitar trabajar con archivos muy pesados, se genera un nuevo dataframe (df)
#en lugar de decirle "lee un nuevo csv" e doy la información con . y ""
new_row=hosp.createDataFrame([("jOSePH mcDOnald",40)],["Name","Age"])

In [ ]:
#una vez tengo el dataframe, luego lo uni. Últil para crearlo, que lo rellene el usuario, y después unirlo
df_joined=df.join(new_row, on="Name", how="right").show()
#en New Row tengo joseph mcdonald, y en df también tengo al mismo señor (tenemos 2), para fusionarlo usamos how (how fusiono?), tenemos right, left, ineer o outer. COn right todos aquellos nombres (porque hemos puesto name) lo va a copiar del df al new row. Si fuese left lo copia de new row a df
#con on le digo el tipo del filtro que estoy aplicando, en este caso Nombre, saca coincidencias exactas en nombres, Lo que digo es: cuando encuentres una coincidencia exacta de nombre vas a copiar el mismo nombre a derecha

#Different columns name
df_joined=df.join(new_row, on=[df["Name"]==new_row["Name"]],how="inner").show()
#si pongo .show(70) me enseña 70 líneas

In [ ]:
#Union of two different DataFrame with identical schemas

data = [("jOSePH mcDOnald", 17, "Male", "AB+", "Diabetes", "2023-01-12",
         "Monica Stone", "Newman-Donovan", "UnitedHealthcare",
         41286.46146554511, 450, "Emergency", "2023-02-10",
         "Lipitor", "Inconclusive")]

# 3. Definir los nombres de las columnas
columns = ["Name", "Age", "Gender", "BloodType", "Condition", "AdmissionDate",
           "Doctor", "Hospital", "Insurance", "Cost", "RoomNumber",
           "VisitType", "DischargeDate", "Medication", "TestResult"]
 #esto lo ha conseguido copiando y pegando los encabezados de la tala (la anterior)

# 4. Crear el DataFrame
new_row = hosp.createDataFrame(data, columns)
#primero le doy la información que quiero contener y después los encabezados. En la anterior poníamos como encabezados schema.head, porque el csv ya tenía encabezados, esto no lo tiene, el encabezado es el nombre de las columnas que he establecido con los 2 comandos de arriba

df_union=df.union(new_row).show()

#otra forma que tenemos para unir filas es el comando .union, permite ir metiendo diferentes filas a mi ds (dataset), para que no de error necesitamos que las columnas del df que cree tienen que tener exactamente el mismo nombre, copiar y pegar
#aquí hemos incorporado la nueva línea a mi data set df, creando df_union

In [ ]:
# no confundir dataframe con rdd, df no solo contiene archivos sino órdenes que he ejecutado (df_union contiene una operación). ahora vamos a trabajar con rdds
df_rdd=df.rdd
df_rdd.collect()

#df tiene información en tablas, y en rdd tiene almacenamiento clave-valor. Ahora podemos hacer mapReduce con mucha facilidad

In [ ]:
# Convertir DataFrame a RDD
df_rdd = df.rdd

# Map: (Hospital, Billing Amount)
mapped_rdd = df_rdd.map(lambda row: (row['Gender'], row['Billing Amount']))
#voy a aplicar el comando MapReduce, que mapea la clave y sacar los valores, creo otro rdd que va a contener el rdd del df (df_rdd) pero con .map que permite mapear. Lambda es el nombre de una función temporal significa que, una vez se ejecutan estas órdenes, lambda desaparece. Si quiero crear una función estable tengo que usar dff o pandas.
#row: quiero que el mapeo sea por filas utilizando la columna género y cantidad de dinero

# Reduce: sumar los Billing Amount por Hospital
reduced_rdd = mapped_rdd.reduceByKey(lambda x, y: x + y)
#una vez me lo ha mapeado y ha establecido el shuffling, agrupo las diferentes claves en los diferentes clústeres. Y a continuación hago la operación de reduce generando un nuevo rdd_reduced utilizando el rdd de mapeo aplicando una reducción por clave. Lambda como función temporal y trabajando con valores x.y, y en este caso necesito que me los sume

# Mostrar resultados
for hospital, total in reduced_rdd.collect():
    print(f"{hospital}: {total}")

#este es un comando para mostrar la operación

#el resultado es el dinero que se han dejado en hombres y en mujeres

In [ ]:
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import FloatType

@pandas_udf(FloatType())
def years_to_days(age_series):
    return age_series *365
#pandas es útil para generar funciones para usar en los rdds, y es más fácil que con pyspark, las librerías de pandas tiene más cosas y no solo se utiliza para BigData entonces es más genérico y más gente contribuye
#esta función puede convertir años en días, interesante porque es una función dinámica: según el usuario meta Nombres, mis valores van a ir cambiando, si calculo el promedio con pandas lo bueno que tiene es que se va a ir actualizando con cada línea que meta el usuario. Esto permite la información en streaming. Esto permite que el modelo predictivo genere datosde manera más actualizada.
#Esto sirve para algoritmos predictivos
#Algoritmos de prediccion: desarollados x gente como FaceBook > predice como se comportan los usuarios y actualizar para optimizar
#Predigo el comportamiento del usuario con software como profet desarrollada por facebook
